# Age & Gender Prediction (TensorFlow/Keras 3)
Compatible with Google Colab.

In [ ]:
!pip -q install -U tensorflow pandas scikit-learn pillow

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print(tf.__version__)

## Prepare DataFrame
Your dataframe must contain columns: `img`, `age`, `gender`. Update the paths as needed.

In [ ]:
IMG_SIZE=(200,200)
BATCH_SIZE=32

def load_image(path):
    img=tf.io.read_file(path)
    img=tf.image.decode_jpeg(img, channels=3)
    img=tf.image.resize(img, IMG_SIZE)
    img=tf.cast(img, tf.float32)/255.0
    return img

def make_dataset(df, shuffle=True):
    paths=df['img'].astype(str).values
    ages=df['age'].astype('float32').values
    genders=df['gender'].astype('float32').values

    ds=tf.data.Dataset.from_tensor_slices((paths, ages, genders))

    def mapper(path, age, gender):
        image=load_image(path)
        return image, {'age_output':age, 'gender_output':gender}

    ds=ds.map(mapper, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds=ds.shuffle(len(df))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Example:
# df=pd.read_csv('/content/labels.csv')
# train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)
# train_ds=make_dataset(train_df)
# test_ds=make_dataset(test_df,shuffle=False)


In [ ]:
from tensorflow.keras import layers, Model

inp=layers.Input(shape=(200,200,3))
x=layers.Conv2D(32,3,activation='relu')(inp)
x=layers.MaxPooling2D()(x)
x=layers.Conv2D(64,3,activation='relu')(x)
x=layers.MaxPooling2D()(x)
x=layers.Conv2D(128,3,activation='relu')(x)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(128,activation='relu')(x)

age=layers.Dense(1,name='age_output')(x)
gender=layers.Dense(1,activation='sigmoid',name='gender_output')(x)

model=Model(inp,[age,gender])

model.compile(
    optimizer='adam',
    loss={'age_output':'mse','gender_output':'binary_crossentropy'},
    metrics={'age_output':['mae'],'gender_output':['accuracy']}
)

model.summary()


In [ ]:
# Train
# history=model.fit(
#     train_ds,
#     validation_data=test_ds,
#     epochs=10
# )


### Why this fixes your error

This notebook uses `tf.data.Dataset` instead of `ImageDataGenerator(..., class_mode='multi_output')`, avoiding the `output_signature`/`TypeSpec` error in TensorFlow/Keras 3.
